# Case 1: Observational Causal Modeling: Activity and Blood Pressure

## Set-up

Imagine you want to give high-quality, data-driven advice to individuals about their activity levels.

You have access to a relatively large retrospective dataset that includes information on

- Detailed dietary intake,
- Physical activity measures,
- Demographics and socioeconomic characteristics,
- Health outcomes such as blood pressure and BMI.

The data are stored in a file named "food-health.csv" (The second code block below downloads this data directly from dropbox.) The associated data dictionary is provided in "data_dictionary.pdf."

---

Suppose you are specifically interested in providing advice around the number of days per week that individuals participate in vigorous physical activity,

    sport_days

and how this relates to systolic blood pressure,

    bp_systolic

You begin by estimating the regression


$$\text{bp_systolic}_i = \alpha + \beta \cdot \text{sport_days}_i + \varepsilon_i$$


to understand the association between vigorous activity and systolic blood pressure.

Let's start there.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
# Get data from dropbox
# Import data from dropbox
!wget -O food_health.csv https://www.dropbox.com/scl/fi/a3jpecmxq1i9tkdesjwtp/food_health.csv?rlkey=ylj48ghu4lvxi89lvupn8k2xn&dl=0

# Import
data = pd.read_csv("food_health.csv")
print(data.head())

In [ ]:
# Summary statistics for two baseline variables
print(data[['bp_systolic', 'sport_days']].describe())

# Need to clean out invalid values (sport_days = 99)
data = data[data['sport_days'] < 99]

# There are also missing values in bp_systolic
# The count is less than the total sample size
data = data[~data['bp_systolic'].isna()]

# Let's check
print('\n')
print(data[['bp_systolic', 'sport_days']].describe())

In [ ]:
# Simple regression
X = sm.add_constant(data['sport_days'])
y = data['bp_systolic']

model = sm.OLS(y, X).fit(cov_type='HC3')
print(model.summary())

## Set-up Questions:

**Question 1.** What is the estimated coefficient on `sport_days`? What does it mean mechanically? Is the relationship between `sport_days` and `bp_systolic` statistically significant?

What does the coefficient estimate imply about whether you should recommend increasing vigorous exercise to individuals who wish to lower their systolic blood pressure? Explain.

(HINT: In responding, be sure to think about the fact that the data is observational. Individuals were not randomly assigned different exercise levels.)

---

**Question 2.** The last line in the regression output reads

> Standard Errors are heteroscedasticity robust (HC3).

Briefly explain what that means and why it is likely important if our goal is to take the statistical conclusion seriously.

(HINT: Any AI can answer this question. It is worth understanding.)




## Main Question Prompt

Your goal is now to do the best you can, using these data, to understand the *causal* effect of vigorous activity (as measured by `sport_days`) on health outcomes. You may focus on `bp_systolic` or choose another outcome (or combination of outcomes). You should justify your choice.

Start by examining the available variables and postulating a directed acyclic graph (DAG) that would allow you to learn the causal effect of `sport_days` on your chosen outcome by conditioning on an appropriate subset of the observed variables.

You should provide a rational argument for why your proposed DAG is a sensible starting point. Your argument does not need to be watertight (it won't be, given the data you have), but it should be logically coherent. For example:

*   Do not condition on outcomes.
*   Justify any included arrows.
*   Justify why certain arrows are excluded.
*   Use sensible groupings of variables where appropriate (e.g., it is not necessary to include a separate node for every food item if you decide that "diet" can be treated as a grouped input).

---

**Question 3.** Provide a drawing of your DAG and explain the underlying rationale.

---

You should now use debiased machine learning (DML) to estimate a partially linear model coefficient in which your chosen outcome is the dependent variable and `sport_days` is the treatment variable, controlling for confounding variables as indicated by your DAG.

---

**Question 4.** Clean the data.

We resolved two data quality issues in our setup. Before estimating any further models, inspect any variables that you intend to use and resolve any issues that you find.

Carefully document everything you do at this stage (e.g. dropping observations, imputing missing variables, variable transformations).

If you do any imputations or transformations, be sure that you do so in a way that avoids data leakage that could contaminate the DML estimation procedure.

---

**Question 5.** Estimate the partially linear regression coefficient using DML.

DML involves building predictive models and cross-fitting. Explain:

* What models were considered.
* How cross-fitting folds were constructed.
* What your final models are
* Why you made these choices

Provide the final estimated effect, its standard error, and explain how to interpret the result.

---

**Question 6.** Finally, and most importantly, build an explanation that you could present to a group of clinicians.

Imagine the clinicians care about giving data-driven advice to patients but are uncomfortable with providing advice they don't understand or can't explain. That is, your explanation should

* Clearly state the conclusion.
* Describe the assumptions under which your estimates have a causal interpretation.
* Discuss important confounders (if any) identified in the prediction steps of DML.
* Highlight relevant caveats.

One natural caveat to address is *heterogeneity*: up to this point, the prompt has asked for a single summary effect. Be prepared to discuss whether the effect may differ across different (groups of) individuals and, if time permits, explore some potential sources of heterogeneity.

This part of the solution should be provided as a slide deck (and notes) suitable for a 10-minute presentation.

---
---
---
## Case 2: Scaling A/B Tests Under Uncertainty

Imagine you are part of the experimentation team at an e-commerce firm that runs a large number of A/B tests each year. Each experiment compares a treatment (a product, pricing, messaging, or user-experience change) to a control and reports summary results from the test. The firm cannot treat every positive-looking result as a guaranteed win, and the stakes vary across experiments: some changes are small and low-risk, while others affect large parts of the customer experience and could have much larger upside (or downside).

In this case, you will work with a synthetic dataset of several thousand experiments. For each experiment, you are given standard A/B test summary statistics, including sample average profit in the control and treatment arms, sample standard deviations, sample sizes, standard errors, the standard error of the difference in means, and a p-value for testing whether the average treatment effect is zero. You will also see additional descriptors of the experiment (for example, the type of platform feature being tested) and a projected future deployment scale, which captures how many customers would be affected if the treatment were rolled out.

Specifically, the variables in the data are

* `experiment_id`: Unique identifier for the experiment.
* `dataset_split`: Indicates which dataset split the row belongs to. You can safely ignore this variable.
* `experiment_area`: Broad area of the business/platform affected by the experiment.
* `intervention_type`: Type of intervention being tested (for example, small UI tweak, major UX change, pricing rule change, recommendation change, etc.).
* `target_segment`: Customer segment targeted by the experiment (for example, new users, returning users, high-value users, etc.).
* `ux_disruption_score`: A numeric score capturing how disruptive the treatment is to the user experience (higher values indicate a larger or more noticeable change).
* `implementation_cost_index`: A numeric index for the cost/complexity of implementing the treatment at scale (higher values indicate more costly implementation). **Important: All outcomes are already net of costs.**
* `team_owner`: Team responsible for the experiment (for example, Growth, Checkout, Search, CRM, etc.).
* `experiment_month`: Month in which the experiment was run.
* `analyst_tenure_months`: Tenure of the analyst running the experiment, measured in months.
* `control_mean_profit`: Sample average profit per customer in the control group. **Important: This IS profit. Costs have been incorporated.**
* `treatment_mean_profit`: Sample average profit per customer in the treatment group.  **Important: This IS profit. Costs have been incorporated.**
* `control_sd_profit`: Sample standard deviation of profit per customer in the control group.
* `treatment_sd_profit`: Sample standard deviation of profit per customer in the treatment group.
* `n_control`: Number of observations (customers) in the control group.
* `n_treatment`: Number of observations (customers) in the treatment group.
* `se_control_mean`: Standard error of the control-group sample mean.
* `se_treatment_mean`: Standard error of the treatment-group sample mean.
* `diff_in_means`: Estimated average treatment effect (ATE), computed as treatment_mean_profit - control_mean_profit.
* `se_diff_in_means`: Standard error of diff_in_means (used for inference on the ATE).
* `p_value_ate_zero`: p-value for testing the null hypothesis that the average treatment effect is zero.
* `future_deploy_scale`: Projected number of customers affected if the treatment is rolled out (used to think about the impact of scaling the treatment).

Your ultimate goal is not just to identify “statistically significant” experiments. Instead, the goal is to think carefully about what a firm should consider when deciding which experiments to scale. In other words: how should a decision rule use estimated effects, uncertainty, scale, and other experiment features when making implementation decisions?

---
### Setup Instructions

For this notebook to work smoothly, you will need the following files in one folder:

- `student_train_experiments.csv`
- `student_practice_eval_experiments.csv`
- `student_final_eval_experiments.csv`
- `evaluator_client.py`


If you are working in **Google Colab**, first put the required files in a single folder in your Google Drive (for example, `MyDrive/MBA_ML/`).

If you are running locally instead of Colab, place the same four files in the **same folder as this notebook** (or update the path in the code cell below).

The code cell below:
1. Mounts Google Drive (if using Colab),
2. Sets the folder containing the files,
3. Verifies that all required files are present, and
4. Makes `evaluator_client.py` importable for the rest of the notebook.

In [ ]:
# --- Setup: file access (Colab + local) ---

from pathlib import Path
import sys

# Toggle this if running locally instead of Colab
RUNNING_IN_COLAB = True

if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # CHANGE THIS if your folder has a different name/location
    BASE_DIR = Path("/content/drive/MyDrive/MBA_ML")
else:
    # Local option 1: same folder as notebook (recommended)
    BASE_DIR = Path(".").resolve()

    # Local option 2: manually set a folder path
    # BASE_DIR = Path(r"C:\Users\YourName\Documents\MBA_ML")

required_files = [
    "student_train_experiments.csv",
    "student_practice_eval_experiments.csv",
    "student_final_eval_experiments.csv",
    "evaluator_client.py",
]

# Check that all files exist
missing = [f for f in required_files if not (BASE_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required file(s) in {BASE_DIR}:\n" + "\n".join(f" - {m}" for m in missing)
    )

# Add BASE_DIR to Python path so we can import evaluator_client.py
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Quick import test
from evaluator_client import practice_evaluate, final_evaluate

print("Setup complete.")
print(f"Using files from: {BASE_DIR}")
print("You can now load the data and call practice_evaluate(...) / final_evaluate(...).")

In [ ]:
# Actually load the data
import pandas as pd

train_df = pd.read_csv(BASE_DIR / "student_train_experiments.csv")
practice_df = pd.read_csv(BASE_DIR / "student_practice_eval_experiments.csv")
final_df = pd.read_csv(BASE_DIR / "student_final_eval_experiments.csv")

print("Training data shape:", train_df.shape)
print("Practice eval shape:", practice_df.shape)
print("Final eval shape:", final_df.shape)

train_df.head(3)

---
### Part 1: Two Simple Implementation Rules

We will begin with two very simple implementation rules.

Consider the following rules:

1. **Rule 1 (Estimated-profit rule):** Implement the treatment if the estimated average treatment effect is positive, i.e. if `diff_in_means > 0`.

2. **Rule 2 (Estimated-profit + significance rule):** Implement the treatment if the estimated average treatment effect is positive **and** the p-value for testing no treatment effect is below 0.05, i.e. if `diff_in_means > 0` and `p_value_ate_zero < 0.05`.

---

**Question 1.**

Provide a short justification for why a firm might want to consider each of these rules. Your answer should explain what information each rule uses and what each rule is implicitly prioritizing. In addition, discuss what assumptions these rules are making when they use experimental results to make rollout decisions. In particular, what would need to be true for the estimated treatment effect from the experiment to be a good guide to what happens after the treatment is deployed more broadly?

---


### Build and Evaluate the Two Simple Rules (Practice Evaluator)

You will now implement the two simple decision rules and evaluate them using the **practice evaluator**.

**Important (credentials):** To use the evaluator, your group needs a **team name** and a **team key/token**. Get these from the instructors (me or the TAs). Do **not** share your token with other groups.

**Important (submission limits):** Each group has **32 practice submissions total** to the practice evaluator. The code cell below will submit **both rules**, so running it once will use **2 of your 32 practice submissions**. Be careful not to re-run the cell repeatedly unless you intend to use additional submissions.

In the code below, you will:
1. Load the practice evaluation dataset,
2. Build a submission for **Rule 1** (`diff_in_means > 0`),
3. Build a submission for **Rule 2** (`diff_in_means > 0` and `p_value_ate_zero < 0.05`),
4. Submit both rules to the practice evaluator, and
5. Compare the returned scores.

The evaluator enforces a 10-second delay between submissions to prevent spam, and the example code includes a short pause so you do not accidentally trigger the cooldown.

In [ ]:
# --- Part 1: Evaluate two simple implementation rules on the practice evaluator ---
# WARNING: Running this cell sends TWO practice submissions (one for each rule).

import pandas as pd
import time

# Assumes BASE_DIR and evaluator_client imports were set up in the earlier setup cell:
# from evaluator_client import practice_evaluate

# 1) Load practice evaluation data
practice_df = pd.read_csv(BASE_DIR / "student_practice_eval_experiments.csv")

# 2) Enter your group credentials (get these from the instructor / TAs)
# TEAM_ID = "GROUP_NAME"              # <-- replace with your team name
# TEAM_TOKEN = "PASTE_TEAM_TOKEN"     # <-- replace with your token
# Example
TEAM_ID = "team_1"
TEAM_TOKEN = "RGNEa_zgsvhSCU3xqegCqg"

BASE_URL = "https://mba-ml-evaluator.onrender.com"  # <-- evaluator URL

# 3) Build Rule 1 submission: implement if estimated ATE > 0
rule1_implement = (practice_df["diff_in_means"] > 0).astype(int)

rule1_submission = pd.DataFrame({
    "experiment_id": practice_df["experiment_id"],
    "implement": rule1_implement
})

# 4) Build Rule 2 submission: implement if estimated ATE > 0 AND p-value < 0.05
rule2_implement = (
    (practice_df["diff_in_means"] > 0) &
    (practice_df["p_value_ate_zero"] < 0.05)
).astype(int)

rule2_submission = pd.DataFrame({
    "experiment_id": practice_df["experiment_id"],
    "implement": rule2_implement
})

# 5) Submit to practice evaluator (this uses 2 practice submissions total)
print("Submitting Rule 1...")
rule1_result = practice_evaluate(
    submission=rule1_submission,
    team_id=TEAM_ID,
    token=TEAM_TOKEN,
    base_url=BASE_URL,
    verbose=True
)

# Enforce a time delay
print("\nWaiting 11 seconds to respect evaluator cooldown...")
time.sleep(11)

print("\nSubmitting Rule 2...")
rule2_result = practice_evaluate(
    submission=rule2_submission,
    team_id=TEAM_ID,
    token=TEAM_TOKEN,
    base_url=BASE_URL,
    verbose=True
)

# 6) Compare results side-by-side
comparison = pd.DataFrame([
    {
        "rule": "Rule 1: ATE > 0",
        "n_implemented": rule1_result["n_implemented"],
        "deployment_rate": rule1_result["deployment_rate"],
        "stationary_total_profit": rule1_result["score_stationary_total"],
        "nonstationary_total_profit": rule1_result["score_nonstationary_total"],
        "avg_profit_per_exp_stationary": rule1_result["avg_profit_per_experiment_stationary"],
        "avg_profit_per_exp_nonstationary": rule1_result["avg_profit_per_experiment_nonstationary"],
    },
    {
        "rule": "Rule 2: ATE > 0 and p < 0.05",
        "n_implemented": rule2_result["n_implemented"],
        "deployment_rate": rule2_result["deployment_rate"],
        "stationary_total_profit": rule2_result["score_stationary_total"],
        "nonstationary_total_profit": rule2_result["score_nonstationary_total"],
        "avg_profit_per_exp_stationary": rule2_result["avg_profit_per_experiment_stationary"],
        "avg_profit_per_exp_nonstationary": rule2_result["avg_profit_per_experiment_nonstationary"],
    }
])

comparison

### Interpreting the Evaluator Output

The table above summarizes how each implementation rule performed in the **practice evaluator**.

The columns are:

- `rule`: The implementation rule that was submitted.
- `n_implemented`: Number of experiments the rule chose to scale up (i.e., number of rows with `implement = 1`).
- `deployment_rate`: Fraction of experiments the rule chose to scale up.
- `stationary_total_profit`: Total incremental profit from the rule in a setting where treatment performance at rollout matches treatment performance in the experiment.
- `nonstationary_total_profit`: Total incremental profit from the rule in a setting where rollout performance can differ from experimental performance (for example, due to scaling frictions, changing conditions, or other real-world effects).
- `avg_profit_per_exp_stationary`: Average incremental profit per experiment under the stationary setting.
- `avg_profit_per_exp_nonstationary`: Average incremental profit per experiment under the nonstationary setting.

You do **not** need to know the details of how the evaluator generates the stationary and nonstationary outcomes. For now, it is enough to interpret them as two different environments for judging the same decision rule: one in which experimental results carry over directly to rollout, and one in which rollout performance may differ.

---

**Question 2.**

Interpret the results. How do the two rules differ in how aggressively they deploy treatments, and how does that affect performance? Why is it interesting to evaluate the same rule under both the stationary and nonstationary scenarios?

## Main Question Prompt



## Main Question Prompt

Your goal is now to design the best implementation rule you can for deciding which experimental treatments the firm should scale.

The firm runs many A/B tests and must decide which treatments to roll out more broadly. A good decision rule should use the information in the experimental summaries and experiment descriptors to make implementation decisions that perform well when evaluated out of sample.

There is no single correct approach. A strong solution will require making choices, justifying them, and testing alternatives. You should expect to go beyond what we covered directly in lecture (using group discussion, AI tools, and outside research as needed).

At a high level, your task is to build a rule that maps each experiment to an implementation decision (`implement = 1` or `0`) and then evaluate that rule using the practice evaluator before choosing a final approach.

---

**Question 3.**

Before building a more advanced rule, explain how you think the firm should evaluate implementation decisions.

Your answer should address:

* What the firm is trying to optimize (for example, total profit, profit per deployed customer, or some balance of upside and downside).
* Why statistical (sampling) uncertainty should or should not matter in the implementation rule.
* Why `future_deploy_scale` may matter for decision-making.
* How you think about differences between experimental performance and rollout performance.

You do not need a formal model, but your reasoning should be clear and logically coherent.

---

**Question 4.**

Develop and compare a set of candidate implementation rules that go beyond the two simple baseline rules from Part 1.

At a minimum, your candidates should include some rules that differ in how they use:

* Estimated treatment effects (`diff_in_means`)
* Statistical uncertainty (for example, standard errors, p-values, confidence bounds, or related ideas)
* Scale (`future_deploy_scale`)
* Experiment descriptors (for example, `experiment_area`, `intervention_type`, `target_segment`, `ux_disruption_score`, `implementation_cost_index`, etc.)

As you build candidate rules, you should explicitly consider whether methods from earlier in the course can help. For example, you might use:

* **Supervised learning** to predict which experiments are more likely to perform well at rollout (or to predict a score that helps rank implementation decisions).
* **Unsupervised learning** to identify clusters/types of experiments and then apply different implementation rules across clusters.

You do **not** have to use both supervised and unsupervised learning, and you do not need to use complex models. You should at least consider whether these tools could help and justify your choices.

If you use machine learning, explain:
* What target (if any) you are trying to predict,
* What features you use,
* How the model output maps into an implementation decision, and
* Why the approach makes sense for this decision problem.

Be explicit about the alternatives you considered and why.

---

**Question 5.**

Use the practice evaluator to test your candidate rules and compare their performance.

In this part, you should:

* Report the main results from the practice evaluator.
* Compare performance across the stationary and nonstationary scenarios.
* Discuss which rules are more or less aggressive (for example, deployment rate and number implemented).
* Explain the tradeoffs you observe (for example, upside vs. conservatism, sensitivity to uncertainty, dependence on scale, etc.).
* Choose a final rule that your group recommends.

Your final rule should be described clearly enough that someone else could reproduce it from your explanation.

---

**Question 6.**

Discuss what assumptions your final rule is relying on and where it could fail.

Your discussion should include:

* What kinds of experiments your rule tends to favor or reject.
* Whether your rule behaves differently across experiment types or target segments.
* What risks the rule is trying to guard against (and what risks it is not guarding against).
* How your conclusions depend on the idea that experimental results may or may not carry over to broader deployment.
* Reflection on takeaways that seem like they could generalize beyond this particular example.

If relevant, include simple diagnostics, subgroup summaries, or plots that help explain how your rule works.

---





### Final Submission (One Attempt Only)

You are now ready to evaluate your **final recommended implementation rule** on the **final evaluation dataset**.

**Important (final submission limit):** Each group may submit to the **final evaluator only once**. There are no retries. Before submitting, make sure your code is finalized and that you have tested your approach using the **practice evaluator**.

**Important (credentials):** Use your group’s assigned `TEAM_ID` and `TEAM_TOKEN` (from the instructors / TAs). Do not share your token.

**Important (class discussion):** We may display a leaderboard of final-evaluation results during the in-class case discussion.

In the code below, you should:

1. Load the final evaluation dataset,
2. Apply your chosen rule to create an `implement` decision (`0` or `1`) for each experiment,
3. Build a submission DataFrame with exactly two columns:
   - `experiment_id`
   - `implement`
4. Submit once to the final evaluator.

Before you run the submission line, double-check:
- Your rule is the one your group wants to use,
- `implement` contains only `0` and `1`,
- You are using the correct team credentials.

In [ ]:
# --- Final submission: ONE final evaluator submission only ---
# WARNING: Running the final_evaluate(...) call below uses your group's ONLY final submission.

import pandas as pd

# Assumes BASE_DIR and evaluator_client imports were set up earlier:
# from evaluator_client import final_evaluate

# 1) Load final evaluation data
final_df = pd.read_csv(BASE_DIR / "student_final_eval_experiments.csv")

# 2) Enter your group credentials (get these from the instructor / TAs)
TEAM_ID = "team_1"                  # <-- replace with your team name
TEAM_TOKEN = "PASTE_TEAM_TOKEN"     # <-- replace with your token
BASE_URL = "https://mba-ml-evaluator.onrender.com"  # <-- evaluator URL

# 3) Apply your FINAL chosen rule to create implement decisions
# ----------------------------------------------------------------
# REPLACE the example rule below with your group's final rule.
# Example placeholder rule:
implement_final = (
    (final_df["diff_in_means"] > 0) &
    (final_df["p_value_ate_zero"] < 0.05)
).astype(int)
# ----------------------------------------------------------------

# 4) Build submission (must contain exactly experiment_id and implement)
final_submission = pd.DataFrame({
    "experiment_id": final_df["experiment_id"],
    "implement": implement_final
})

# Optional safety checks (recommended)
assert set(final_submission.columns) == {"experiment_id", "implement"}
assert final_submission["experiment_id"].is_unique
assert final_submission["implement"].isin([0, 1]).all()
assert len(final_submission) == len(final_df)

print("Final submission preview:")
display(final_submission.head())
print("\nCounts:")
print(final_submission["implement"].value_counts(dropna=False).sort_index())

# 5) FINAL submission (UNCOMMENT ONLY WHEN READY)
# final_result = final_evaluate(
#     submission=final_submission,
#     team_id=TEAM_ID,
#     token=TEAM_TOKEN,
#     base_url=BASE_URL,
#     verbose=True
# )

# If you submit successfully, you may want to save the returned result:
# print(final_result)

**Question 7.**
Finally, and most importantly, build an explanation that you could present to a group of business decision makers.

Imagine the audience is willing to use data-driven experimentation but is not interested in technical details unless they affect the decision. Your presentation should:

* Clearly state your recommended implementation rule.
* Explain the logic of the rule in plain language.
* Summarize how the rule performs in the practice evaluation.
* Highlight the difference between stationary and nonstationary performance and why that matters.
* Discuss the main limitations/caveats and what additional information the firm might want before scaling decisions.
* While it would not be something in a real presentation, you should also discuss your performance on the final evaluation (and how it compares) to your practice runs (assuming you did them).

This part of the solution should be provided as a slide deck (and notes) suitable for a 10-minute presentation.